In [3]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.datasets import imdb
import numpy as np
import keras_tuner as kt

# ── Data ──────────────────────────────────────────────────────────────────────
max_words = 10000
maxlen    = 200

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_words)

# Real binary labels (0 = Negative, 1 = Positive) — NO random assignment
# y_train and y_test are already 0/1, so we use them directly
x_train = pad_sequences(x_train, maxlen=maxlen)
x_test  = pad_sequences(x_test,  maxlen=maxlen)

# ── Model builder ─────────────────────────────────────────────────────────────
def build_model(hp):
    model = Sequential([
        Embedding(max_words,
                  hp.Int('embedding_dim', min_value=32, max_value=256, step=32)),
        LSTM(hp.Int('lstm_units', min_value=32, max_value=128, step=32),
             dropout=hp.Float('dropout', min_value=0.0, max_value=0.5, step=0.1),
             recurrent_dropout=hp.Float('rec_dropout', min_value=0.0, max_value=0.5, step=0.1)),
        Dense(1, activation='sigmoid')          # ← binary output
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
        ),
        loss='binary_crossentropy',             # ← binary loss
        metrics=['accuracy']
    )
    return model

# ── Tuner ─────────────────────────────────────────────────────────────────────
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=4,
    executions_per_trial=1,
    directory='my_dir',
    project_name='imdb_binary_tuning'
)

print("Searching for best hyperparameters...")
tuner.search(x_train, y_train,          # ← real labels, shape (25000,)
             epochs=3,
             validation_split=0.2,
             batch_size=64)

# ── Best hyperparameters ──────────────────────────────────────────────────────
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"\nBest embedding_dim : {best_hps.get('embedding_dim')}")
print(f"Best lstm_units    : {best_hps.get('lstm_units')}")
print(f"Best dropout       : {best_hps.get('dropout')}")
print(f"Best rec_dropout   : {best_hps.get('rec_dropout')}")
print(f"Best learning_rate : {best_hps.get('learning_rate')}")

# ── Evaluate ──────────────────────────────────────────────────────────────────
best_model = tuner.get_best_models(num_models=1)[0]
loss, accuracy = best_model.evaluate(x_test, y_test, verbose=0)
print(f"\nTest Loss     : {loss:.4f}")
print(f"Test Accuracy : {accuracy:.4f}")

# ── Inference ─────────────────────────────────────────────────────────────────
word_index = imdb.get_word_index()

def predict_sentiment(review_text):
    words    = review_text.lower().split()
    sequence = [word_index.get(w, 0) + 3 for w in words]
    sequence = [i if i < max_words else 2 for i in sequence]
    padded   = pad_sequences([sequence], maxlen=maxlen)

    prob = best_model.predict(padded, verbose=0)[0][0]   # single sigmoid value
    label = "Positive 😊" if prob >= 0.5 else "Negative 😞"
    return label, float(prob)

movie_name = input("Enter a movie name: ")
if movie_name:
    sentiment, confidence = predict_sentiment(movie_name)
    print(f"\nPrediction for '{movie_name}': {sentiment}")
    print(f"Confidence  : {confidence:.2%}")

Trial 4 Complete [00h 12m 38s]
val_accuracy: 0.8697999715805054

Best val_accuracy So Far: 0.8697999715805054
Total elapsed time: 00h 49m 14s

Best embedding_dim : 192
Best lstm_units    : 32
Best dropout       : 0.1
Best rec_dropout   : 0.1
Best learning_rate : 0.01


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))



Test Loss     : 0.3369
Test Accuracy : 0.8574
1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step
Enter a movie name: the matrix

Prediction for 'the matrix': Positive 😊
Confidence  : 65.60%


In [2]:
pip install keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 11.5 MB/s eta 0:00:00
